# Session 8: Exercises

**Before session 9.** Predict, then measure. Answers go in the **✍️** cells, and code TODOs are
marked `# TODO`.

Nothing here calls a chat model. Exercise 1 needs no model at all: it is decided by where the
splitter cut. The others embed text with `gemini-embedding-2` on the free tier (or the fastembed
fallback, section 9 of the notebook). Exercises 2 and 3 read the real knowledge base, so run
`uv run python kb/ingest.py` from `project/` first, and close the notebook's kernel if it still holds
the folder. Exercises 4 and 5 build throwaway in-memory collections and leave it untouched.

## Setup: given, just run it

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")   # fastembed's progress bar

from langchain.embeddings import init_embeddings                    # noqa: E402
from langchain_core._api import LangChainBetaWarning                # noqa: E402

warnings.filterwarnings("ignore", category=LangChainBetaWarning)

HERE = Path.cwd()
sys.path.insert(0, str(HERE / "project"))
from config import settings                       # noqa: E402
from kb import ingest, store                      # noqa: E402
from kb import retrieval_eval as scorer           # noqa: E402

emb = store.embeddings()
corpus = ingest.load_corpus()


def lab_store(chunks, name, embeddings=emb, mode="dense"):
    """A throwaway in-memory collection holding `chunks`. The real knowledge base is untouched."""
    client = store.open_client(":memory:")
    store.ensure_collection(client, embeddings, name)
    store.vector_store(client, embeddings, mode="hybrid", name=name).add_documents(
        chunks, ids=[ingest.chunk_id(c) for c in chunks])
    return store.vector_store(client, embeddings, mode=mode, name=name)


def hits(vs, query, k=4, filter=None):
    """Top-k results, one line each."""
    print(query)
    for doc, score in vs.similarity_search_with_score(query, k=k, filter=filter):
        m = doc.metadata
        print(f"  {score:.3f}  {m['source']} · {m['section']:<30} {' '.join(doc.page_content.split())[:70]}")


print(f"{len(corpus)} documents · embedding space {settings.collection}")

## ⭐ Exercise 1: Chunk-size autopsy

**What this teaches:** a retrieval miss can be decided before any search runs, by where the splitter cut.

In the notebook's section 12 table, `chunk_size=200` missed **KBQ-05**: *"I'm a gold member — do I have
to pay for return shipping?"* Find out why without running a single search.

1. Chunk the corpus at 200 (overlap 40).
2. Print every chunk that mentions `gold-tier`, and the chunk that follows it.
3. Say what the scorer was looking for (`evidence` in `data/kb_questions.yml`) and why no chunk has it.

In [ ]:
small = ingest.chunk(corpus, 200, 40)

# TODO: find the index i of each chunk whose text contains "gold-tier",
#       then print small[i].page_content and small[i + 1].page_content.

**✍️ Your answer.** Why did KBQ-05 miss, and what would a customer have been told?

## ⭐⭐ Exercise 2: Add gold questions, and find a miss

**What this teaches:** the gold set is the thing you grow, and a miss tells you which layer to fix.

Write **two** new gold questions in the same shape as `data/kb_questions.yml`: one plain, and one
phrased the way an annoyed customer would phrase it. Score each against the real knowledge base
(dense, k=4). Then find one question — yours, or a variant — that **misses**, and name the layer at
fault: loader, splitter, embedding, or the question's wording.

In [ ]:
client = store.open_client()          # the real knowledge base (run kb/ingest.py first)
dense = store.vector_store(client, emb)
search = lambda q, k: dense.similarity_search(q, k=k)

MINE = [
    # TODO: two entries like the ones in data/kb_questions.yml. Pick an existing clause, e.g.
    # {"id": "MY-01", "question": "...", "source": "shipping-policy", "evidence": ["PO boxes"]},
]

for q in MINE:
    print(q["id"], "rank:", scorer.rank_of(q, search(q["question"], 4)))

client.close()                        # release the folder for ingest.py

**✍️ Your answer.** Which question missed or ranked low, and which layer would you fix first?

## ⭐⭐ Exercise 3: Shop rules vs the regulator

**What this teaches:** a metadata filter decides whose words an answer is allowed to quote.

Ask *"How long does a seller have to ship if they didn't say?"* three ways against the real knowledge
base: no filter, `authority == "regulator"`, and `authority == "shop"`. Then explain why a support bot
must never present the regulator's text as ShopWise's promise.

In [ ]:
client = store.open_client()
dense = store.vector_store(client, emb)
Q = "How long does a seller have to ship if they didn't say?"

# TODO: hits(dense, Q, k=2, filter=...) three times: filter=None,
#       store.where(authority="regulator"), store.where(authority="shop")

client.close()

**✍️ Your answer.** Why must the answer name its authority?

## ⭐⭐⭐ Exercise 4: The wrong-space incident

**What this teaches:** the failure you cannot see: same dimension, different model, confident noise.

Build an in-memory copy of the knowledge base with `gemini-embedding-2` at 768. Score it with the
right query model (dense, k=4). Then score the **same index** while embedding the *questions* with
`gemini-embedding-001` at 768 : the same size, a different model. Nothing will error. Write the
three-sentence incident report a lead would want.

Hint: `scorer.score(search, k)` takes any `search(question, k)` function. Search by vector with
`vs.similarity_search_by_vector(other.embed_query(q), k=k)`.

In [ ]:
chunks = ingest.chunk(corpus)
vs = lab_store(chunks, "wrong_space")
right = scorer.score(lambda q, k: vs.similarity_search(q, k=k), 4)
print(f"right model:  hit-rate@4 {right['hit_rate']:.2f}  MRR {right['mrr']:.2f}")

other = init_embeddings("google_genai:gemini-embedding-001", output_dimensionality=768)
# TODO: score again, embedding each question with `other` and searching by vector.

**✍️ Your answer.** Write the incident report: what happened, why nothing errored, what recovery costs.

## ⭐⭐⭐ [ADV] Exercise 5: A third authority

**What this teaches:** ingesting a real document you did not write, and deciding what to do with its mess.

Bangladesh's *Consumers' Right Protection Act, 2009* has an authentic English text on the Directorate
of National Consumer Rights Protection's site (dncrp.portal.gov.bd). It is **not committed** to this
repo, because no open licence is stated. Download it yourself into the gitignored `.state/`
folder, then:

1. Load it with `ingest.load_pdf` and print the start of pages 1 and 2. What did the loader lose,
   and what did it garble?
2. Give it `authority = "regulator-bd"`, chunk it, and index it into **its own** in-memory collection.
3. Ask it *"What can a consumer do if a seller sells an adulterated product?"* and read the hits.
4. Decide: clean the garbled pages, drop them, or keep them — and justify it in one line.

In [ ]:
PDF = settings.DATA_DIR.parent / ".state" / "consumers-right-protection-act-2009.pdf"
URL = ("https://objectstorage.ap-dcc-gazipur-1.oraclecloud15.com/n/axvjbnqprylg/b/V2Ministry/o/"
       "office-dncrp/2024/12/20be8c7434ec486cb743c5ccc0cf8405.pdf")

# TODO: if not PDF.exists(): download URL to PDF (httpx.get(URL, follow_redirects=True).content)
# TODO: pages = ingest.load_pdf(PDF); set each page's metadata["authority"] = "regulator-bd"
# TODO: chunks = ingest.chunk(pages); vs = lab_store(chunks, "bd_act"); hits(vs, "...")

**✍️ Your answer.** What did the loader garble, and what did you decide to do about it?